*** This code is to test the windy environments ***

In [1]:
# Import libraries
import pygame
import gymnasium as gym
import numpy as np
import copy
import itertools
import math
np.random.seed(33) # seeding

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# convert binary list to decimal
def binary_list_to_decimal(bin_list):
    bin = ''
    for b in bin_list:
        bin += str(b)
    dec = int(bin,2)
    return dec

# Function to check if a point is inside a polygon (Ray-casting algorithm)
def is_inside_polygon(point, poly):
    x, y = point
    inside = False
    n = len(poly)
    p1x, p1y = poly[0]
    for i in range(n + 1):
        p2x, p2y = poly[i % n]
        if min(p1y, p2y) < y <= max(p1y, p2y) and x <= max(p1x, p2x):
            if p1y != p2y:
                xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
            if p1x == p2x or x <= xinters:
                inside = not inside
        p1x, p1y = p2x, p2y
    return inside

# Function to return minimum distance in a list of points
def min_dist(x):
    x = np.array(x).astype('float32')
    dists = []
    for p1, p2 in itertools.combinations(x, 2):
        dist = np.linalg.norm(p1-p2)
        dists.append(dist)
    return float(np.min(dists))

In [3]:
experiments_path = r'./experiment_sets.txt'
# Read the experiments file and select the experiment
with open(experiments_path, 'r') as experiment_file:
    codes = experiment_file.read()
    exec(codes) # execute
selected_experiment = set3 # Select the experiment set
selected_experiment

{'field': [(13.0, 23.0), (27.0, 21.0), (31.0, 32.0), (27.0, 33.0)],
 'init_positions': [array([20., 25.]), array([30., 30.]), array([25., 30.])],
 'infected_locations': {(16.0, 25.0),
  (20.0, 24.0),
  (21.0, 28.0),
  (24.0, 30.0),
  (27.0, 23.0),
  (27.0, 28.0)}}

In [4]:
sf = 10 # scaling factor
selected_experiment['field'] = [(x*sf, y*sf) for (x,y) in selected_experiment['field']]
selected_experiment['infected_locations'] = [(x*sf, y*sf) for (x,y) in selected_experiment['infected_locations']]
selected_experiment['init_positions'] = [v*sf for v in selected_experiment['init_positions']]
selected_experiment, len(selected_experiment['init_positions']), len(np.unique(selected_experiment['init_positions'], axis=0))

({'field': [(130.0, 230.0), (270.0, 210.0), (310.0, 320.0), (270.0, 330.0)],
  'init_positions': [array([200., 250.]),
   array([300., 300.]),
   array([250., 300.])],
  'infected_locations': [(240.0, 300.0),
   (160.0, 250.0),
   (200.0, 240.0),
   (270.0, 280.0),
   (210.0, 280.0),
   (270.0, 230.0)]},
 3,
 3)

# Inference

In [ ]:
class MultiRobotEnv(gym.Env):
    metadata = {'render_modes': ['human', 'print', 'rgb_array'], "render_fps": 4}
    def __init__(self, render_mode=None, field_info=copy.deepcopy(selected_experiment), wind_par=[0,0], num_robots=3):
        super(MultiRobotEnv, self).__init__()
        # Screen dimensions
        self.edge_buffer = 10 # Boundary above the max values
        self.poly_vertices = field_info['field'] # Vertices of polygon
        self.xs, self.ys = zip(*field_info['field']) # x and y values of the vertices of the polygonal field
        self.WIDTH, self.HEIGHT = 1000, 1000 # Use this if we want to have fixed width and height  
        # self.WIDTH, self.HEIGHT = max(self.xs) + self.edge_buffer, max(self.ys) + self.edge_buffer        

        # Robot parameters
        self.num_robots = num_robots # Rendering error if more than 7
        self.init_robot_positions = np.array(field_info['init_positions'])[:self.num_robots]
        self.robot_size = 10
        self.mass = 1.0
        self.thrust_power = 0.5  # Force applied per action
        self.max_speed = 5  # Maximum speed    
        self.min_speed = -5 # Minimum speed
        self.min_positions = np.zeros(self.num_robots*2) # Minimum positions
        self.max_positions = np.array([[self.WIDTH, self.HEIGHT] for _ in range(self.num_robots)]) # Maximum positions
        self.min_velocities = np.array([[self.min_speed, self.min_speed] for _ in range(self.num_robots)]) # Min speed list
        self.max_velocities = np.array([[self.max_speed, self.max_speed] for _ in range(self.num_robots)]) # Max speed list
        self.wind_f_a, self.wind_beta_a = wind_par # Wind parameters: magnitude and angle

        # infected locations
        self.initial_inf_locations = field_info['infected_locations']
        self.infected_size = 10 # Radius of infected locations
        self.infected_length = len(field_info['infected_locations'])
        self.infected_state_length = 2**(self.infected_length) # 2**5, binary to decimal

        # Action space: thrust in x and y directions for each robot
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(self.num_robots, 2), dtype=np.float32)

        # Observation space: position and velocity (x, y, vx, vy) for each robot + infected location        
        self.observation_space = gym.spaces.Box( # The (visited) weed locations are tracked on the observation space
                    low = np.concatenate((self.min_positions.flatten(), self.min_velocities.flatten(), np.array([0]))), # Lowest positions and velocities
                    high = np.concatenate((self.max_positions.flatten(), self.max_velocities.flatten(), np.array([self.infected_state_length - 1]))), # highest positions and velocities
                    dtype=np.float32)

        assert render_mode is None or render_mode in self.metadata["render_modes"] # Check if the render mode is correct
        self.render_mode = render_mode
        self.screen = None
        self.clock = None
        # If human-rendering is used, `self.screen` will be a reference to the screen that we draw to. `self.clock` will be a clock that is used
        # to ensure that the environment is rendered at the correct framerate in human-mode. They will remain `None` until human-mode is used for the first time.   

        # Reset the environment and start
        self.reset()
    
    def _get_obs(self):
        info = {f'robot{i}': self.robot_positions[i] for i in range(self.num_robots)} # Current position of each robot
        infected = binary_list_to_decimal(list(self.infected_dict.values())) # Convert the binary list of infected locations to a decimal value
        state = np.concatenate((self.robot_positions.flatten(), self.robot_velocities.flatten(), np.array([infected])), dtype=np.float32) # Current state of the robots
        return state, info        

    def reset(self, seed=None, options={}):
        # Reset the visited states and counts
        self.step_count = 0
        self.visited = set()
        self.infected_locations = copy.deepcopy(self.initial_inf_locations) # Initial infected locations
        self.infected_dict = {v:0 for v in self.infected_locations} # 0 for unvisited infected locations, 1 for visited
        self.robot_positions = copy.deepcopy(self.init_robot_positions) # Initial positions of each robot
        self.robot_velocities = np.zeros((self.num_robots, 2)) # Initial velocities of each robot (zero)
        return self._get_obs()
    
    def step(self, actions):
        terminated, truncated = False, False
        rewards = 0
        self.step_count += 1
        for i in range(self.num_robots): # For every robot
            ax, ay = actions[i] * self.thrust_power # What actions to take

            # Update velocity
            self.robot_velocities[i][0] += ax / self.mass + self.wind_f_a * np.cos(np.radians(self.wind_beta_a))
            self.robot_velocities[i][1] += ay / self.mass + self.wind_f_a * np.sin(np.radians(self.wind_beta_a))

            # Limit velocity
            self.robot_velocities[i] = np.clip(self.robot_velocities[i], self.min_speed, self.max_speed)

            # Predict new position
            new_position = self.robot_positions[i] + self.robot_velocities[i]

            # Boundary conditions (keep robot within polygon)
            if is_inside_polygon(new_position, self.poly_vertices):
                pass
            else: # Hits the wall!
                rewards -= 10000 # Medium negative reward for hitting the wall
                self.robot_velocities[i][:] = 0 # Stop movement

            # Update position
            self.robot_positions[i] += self.robot_velocities[i]
            
            # Boundary conditions (keep robot within screen)
            self.robot_positions[i] = np.clip(self.robot_positions[i], [0, 0], [self.WIDTH, self.HEIGHT])

            # Check if location is visited before, and add it to the visited locations
            if tuple(self.robot_positions[i]) in self.visited:
                rewards -= 100 # Small negative reward for visiting previous location
            else:
                rewards -= 10 # Very small negative reward for visiting new locations
            self.visited.add(tuple(self.robot_positions[i]))            

            # Check if any infected location is visited        
            nearby_infected_locations = [] # To store the nearby infected locations
            for j, inf_loc in enumerate(self.infected_locations): # Loop through each infected location
                dist = np.linalg.norm(self.robot_positions[i]-inf_loc) # Distance between robot position and infected location
                if dist <= self.infected_size: # If the distance is within the radius of the infected location size
                    nearby_infected_locations.append(inf_loc) # Add the infected location
                    rewards += 10000 # Medium positive rewards for visiting each infected location
                    # input("Pause!") # Only pause if you want to visualize visiting infected locations
            for inf_loc in nearby_infected_locations:
                self.infected_locations.remove(inf_loc) # Delete each visited infected location
                self.infected_dict[tuple(inf_loc)] = 1 # Update the infected dictionary
        
        # Check if all infected locations are visited
        if len(self.infected_locations) == 0:
            rewards += 100000 # Big positive rewards for visiting all infected locations
            terminated = True
        
        # Check if any collisions occurred
        if self.num_robots > 1:
            min_dist_between_robots = min_dist(self.robot_positions) # Minimum distance between robots
            if min_dist_between_robots < self.robot_size:
                rewards = -100000 # Big negative rewards for collisions
                terminated = True

        obs, info = self._get_obs() # Get the updated observations
        # rewards = rewards * self.gamma ** self.step_count
        return obs, rewards, terminated, truncated, info
    
    def render(self):
        # Initialize pygame
        if self.screen is None and self.render_mode == "human": # Initialize pygame if it is not initialized
            pygame.init()
            pygame.display.init()
            self.screen = pygame.display.set_mode((self.WIDTH, self.HEIGHT))
            pygame.display.set_caption("Multi-robot RL Environment")
            if self.clock is None:
                self.clock = pygame.time.Clock()
                self.running = True
        
        self.screen.fill((255, 255, 255)) # White color for the background
        colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 128, 0), (128, 0, 255), (255, 0, 255), (128, 128, 128)]  # Colors for each robot: Red, Green, Blue, Orange, Violet, Pink, Grey
        pix_size = 10

        # Draw the polygon
        # pixel_poly_vertices = [(point[0] * pix_size, point[1] * pix_size) for point in self.poly_vertices]
        pygame.draw.polygon(surface=self.screen, 
                            color=(255, 255, 0), # Yello color for the polygon
                            points=self.poly_vertices)
        
        # Draw the visited regions
        for point in self.visited:
            pygame.draw.circle(self.screen, pygame.Color(100, 100, 100, a=0.2), point, pix_size/2) # Light grey color for visited regions, with transparency alpha

        # Draw robots
        for i in range(self.num_robots):
            pygame.draw.circle(self.screen, colors[i], (int(self.robot_positions[i][0]), int(self.robot_positions[i][1])), pix_size/2) # Pick the colors from above list

        # Draw infected locations
            for l in self.infected_locations:
                pygame.draw.circle(self.screen, (0, 255, 255), (int(l[0]), int(l[1])), pix_size/2) # Cyan color for infected locations
        
        pygame.display.flip() # Allows only a portion of the screen to be updated
        self.clock.tick(60)
    
    def close(self):
        if self.screen is not None:
            pygame.display.quit()
            pygame.quit()

In [6]:
# Register environment
gym.register(id='MultiRobotEnv-v0', 
             entry_point=MultiRobotEnv,
             max_episode_steps=1000)

Load trained network:

In [7]:
# from sb3_contrib import TRPO

# weights_path = rf"C:\Users\choton\OneDrive - Kansas State University\PhD Projects\Reinforcement Learning\Codes\for_coRL\github\FlowBotic\trained_models\new_mar25_env1_trpo.zip"

# # Load trained network
# model = TRPO.load(weights_path)

In [8]:
from sb3_contrib import CrossQ

weights_path = rf"C:\Users\choton\OneDrive - Kansas State University\PhD Projects\Reinforcement Learning\Codes\for_coRL\github\FlowBotic\trained_models\new_mar25_env3_CrossQ.zip"

# Load trained network
model = CrossQ.load(weights_path)

Play using trained network and default env (we can also use vector env):

In [21]:
def play(wind_par):
    global env
    # Make the environment
    env = gym.make('MultiRobotEnv-v0', render_mode='human', wind_par=wind_par)
    env.metadata['render_fps'] = 1
    obs, info = env.reset()
    env.render()
    pygame.event.get()

    # Start playing
    terminated, truncated = False, False
    total_rewards = 0
    total_steps = 0
    while True:
        action, _ = model.predict(obs)
        print(action)
        # print(int(action))
        obs, reward, terminated, truncated,  info = env.step(action)
        env.render()
        total_rewards += reward
        print(f"Obs: {obs}, Reward: {reward}, terminated: {terminated}, total_rewards: {total_rewards}, total_steps: {total_steps}")
        if terminated or truncated:
            print('terminated:', terminated, 'truncated:', truncated)
            break
        pygame.event.get()
        total_steps += 1

In [10]:
assert False, "Play one by one"

AssertionError: Play one by one

In [22]:
play(wind_par=[0,0])

c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\gymnasium\spaces\box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(


[[ 0.9778676  -0.85708964]
 [ 0.06870162 -0.07848513]
 [-0.9872851   0.7256397 ]]
Obs: [ 2.0048894e+02  2.4957146e+02  3.0003436e+02  2.9996075e+02
  2.4950636e+02  3.0036282e+02  4.8893380e-01 -4.2854482e-01
  3.4350812e-02 -3.9242566e-02 -4.9364254e-01  3.6281985e-01
  4.0000000e+01], Reward: 19970, terminated: False, total_rewards: 19970, total_steps: 0
[[ 0.841015   -0.9559003 ]
 [-0.9355316  -0.69875395]
 [-0.9999079  -0.7178792 ]]
Obs: [ 2.0139838e+02  2.4866496e+02  2.9960092e+02  2.9957214e+02
  2.4851276e+02  3.0036670e+02  9.0944129e-01 -9.0649498e-01
 -4.3341500e-01 -3.8861954e-01 -9.9359649e-01  3.8802624e-03
  4.0000000e+01], Reward: -30, terminated: False, total_rewards: 19940, total_steps: 1
[[ 0.8846655  -0.46344244]
 [-0.95967114  0.5486531 ]
 [-0.99926895 -0.99172574]]
Obs: [ 2.0275015e+02  2.4752675e+02  2.9868768e+02  2.9945786e+02
  2.4701953e+02  2.9987473e+02  1.3517740e+00 -1.1382163e+00
 -9.1325057e-01 -1.1429298e-01 -1.4932309e+00 -4.9198261e-01
  4.0000000e+0

In [15]:
play(wind_par=[0.1,30])

[[ 0.9774234  -0.9023483 ]
 [ 0.22726953 -0.17495674]
 [-0.9890674   0.78502905]]
Obs: [ 2.0057532e+02  2.4959883e+02  3.0020023e+02  2.9996252e+02
  2.4959207e+02  3.0044250e+02  5.7531428e-01 -4.0117413e-01
  2.0023730e-01 -3.7478369e-02 -4.0793115e-01  4.4251454e-01
  4.0000000e+01], Reward: 19970, terminated: False, total_rewards: 19970, total_steps: 0
[[ 0.88938594 -0.9457635 ]
 [-0.800158   -0.8901093 ]
 [-0.99961036 -0.9809999 ]]
Obs: [ 2.0168193e+02  2.4877477e+02  3.0008701e+02  2.9953000e+02
  2.4877094e+02  3.0044452e+02  1.1066098e+00 -8.2405591e-01
 -1.1323917e-01 -4.3253303e-01 -8.2113379e-01  2.0145834e-03
  4.0000000e+01], Reward: -30, terminated: False, total_rewards: 19940, total_steps: 1
[[ 0.88533425 -0.96700305]
 [-0.9624218  -0.12910378]
 [-0.9985537  -0.9726892 ]]
Obs: [203.31781    247.51721    299.57916    299.08292    247.53712
 300.0102       1.6358794   -1.2575574   -0.5078475   -0.4470849
  -1.233808    -0.43433002  40.        ], Reward: -30, terminated: Fa

In [16]:
play(wind_par=[0.2,30])

[[ 0.9652903  -0.9185201 ]
 [-0.3051334   0.02869332]
 [-0.9806366   0.8159462 ]]
Obs: [ 2.0065585e+02  2.4964075e+02  3.0002063e+02  3.0011435e+02
  2.4968289e+02  3.0050797e+02  6.5585023e-01 -3.5926005e-01
  2.0638380e-02  1.1434666e-01 -3.1711322e-01  5.0797313e-01
  4.0000000e+01], Reward: 19970, terminated: False, total_rewards: 19970, total_steps: 0
[[ 0.84004307 -0.95442134]
 [-0.94939274 -0.9761612 ]
 [-0.99961    -0.86340237]]
Obs: [ 2.0190492e+02  2.4890427e+02  2.9973978e+02  2.9984061e+02
  2.4903917e+02  3.0068423e+02  1.2490768e+00 -7.3647070e-01
 -2.8085291e-01 -2.7373394e-01 -6.4371312e-01  1.7627193e-01
  4.0000000e+01], Reward: -30, terminated: False, total_rewards: 19940, total_steps: 1
[[ 0.9749789  -0.84659934]
 [-0.9341865  -0.07087678]
 [-0.99969983 -0.9374707 ]]
Obs: [ 2.0381470e+02  2.4784450e+02  2.9916504e+02  2.9963144e+02
  2.4806882e+02  3.0049179e+02  1.9097714e+00 -1.0597703e+00
 -5.7474107e-01 -2.0917232e-01 -9.7035795e-01 -1.9246341e-01
  4.0000000e+0

In [17]:
play(wind_par=[0.3,30])

[[ 0.98695683 -0.90903145]
 [-0.38332534 -0.04336536]
 [-0.9901163   0.66757274]]
Obs: [ 2.0075328e+02  2.4969548e+02  3.0006815e+02  3.0012833e+02
  2.4976476e+02  3.0048380e+02  7.5328606e-01 -3.0451572e-01
  6.8144955e-02  1.2831733e-01 -2.3525053e-01  4.8378637e-01
  4.0000000e+01], Reward: 19970, terminated: False, total_rewards: 19970, total_steps: 0
[[ 0.86538506 -0.9434201 ]
 [-0.89239985 -0.9691044 ]
 [-0.9993194  -0.64435995]]
Obs: [ 2.0219907e+02  2.4906926e+02  2.9994989e+02  2.9992209e+02
  2.4928964e+02  3.0079538e+02  1.4457862e+00 -6.2622577e-01
 -1.1824735e-01 -2.0623489e-01 -4.7510260e-01  3.1160641e-01
  4.0000000e+01], Reward: -30, terminated: False, total_rewards: 19940, total_steps: 1
[[ 0.91715837 -0.48054606]
 [-0.94696724 -0.75679064]
 [-0.9995216  -0.80766934]]
Obs: [ 2.0436325e+02  2.4835275e+02  2.9961798e+02  2.9948746e+02
  2.4857458e+02  3.0085315e+02  2.1641729e+00 -7.1649879e-01
 -3.3192337e-01 -4.3463022e-01 -7.1505576e-01  5.7771724e-02
  4.0000000e+0

In [18]:
play(wind_par=[0.4,30])

[[ 0.98889434 -0.77921844]
 [-0.15752006  0.39358675]
 [-0.9818906   0.6522541 ]]
Obs: [ 2.0084085e+02  2.4981039e+02  3.0026764e+02  3.0039679e+02
  2.4985547e+02  3.0052612e+02  8.4085733e-01 -1.8960921e-01
  2.6765013e-01  3.9679337e-01 -1.4453515e-01  5.2612704e-01
  4.0000000e+01], Reward: 19970, terminated: False, total_rewards: 19970, total_steps: 0
[[ 0.8467324  -0.9941571 ]
 [-0.37895954 -0.9642459 ]
 [-0.9997514  -0.8123407 ]]
Obs: [ 2.0245149e+02  2.4932370e+02  3.0069223e+02  3.0051147e+02
  2.4955746e+02  3.0084607e+02  1.6106337e+00 -4.8668775e-01
  4.2458051e-01  1.1467042e-01 -2.9800069e-01  3.1995672e-01
  4.0000000e+01], Reward: -30, terminated: False, total_rewards: 19940, total_steps: 1
[[ 0.74021244 -0.94139427]
 [-0.97302496 -0.9772493 ]
 [-0.99871284 -0.9708994 ]]
Obs: [ 2.0477864e+02  2.4856631e+02  3.0097672e+02  3.0033752e+02
  2.4910652e+02  3.0088058e+02  2.3271501e+00 -7.5738490e-01
  2.8447822e-01 -1.7395425e-01 -4.5094693e-01  3.4507014e-02
  4.0000000e+0

In [19]:
play(wind_par=[0.5,30])

[[ 0.9831462  -0.86420536]
 [-0.09426522 -0.30889112]
 [-0.9959431   0.7961854 ]]
Obs: [ 2.0092459e+02  2.4981790e+02  3.0038589e+02  3.0009555e+02
  2.4993504e+02  3.0064810e+02  9.2458582e-01 -1.8210268e-01
  3.8588008e-01  9.5554441e-02 -6.4958863e-02  6.4809269e-01
  4.0000000e+01], Reward: 19970, terminated: False, total_rewards: 19970, total_steps: 0
[[-0.6745917  -0.95393753]
 [-0.5990405  -0.9954737 ]
 [-0.9997764  -0.6211332 ]]
Obs: [ 2.0194489e+02  2.4940883e+02  3.0090524e+02  2.9994336e+02
  2.4980321e+02  3.0123563e+02  1.0203027e+00 -4.0907145e-01
  5.1937252e-01 -1.5218240e-01 -1.3183437e-01  5.8752608e-01
  4.0000000e+01], Reward: -30, terminated: False, total_rewards: 19940, total_steps: 1
[[ 0.8725352  -0.9989213 ]
 [-0.6817452  -0.9870127 ]
 [-0.9988969  -0.70971334]]
Obs: [ 2.0383447e+02  2.4875029e+02  3.0151675e+02  2.9954767e+02
  2.4960493e+02  3.0171829e+02  1.8895830e+00 -6.5853208e-01
  6.1151266e-01 -3.9568874e-01 -1.9827011e-01  4.8266941e-01
  4.0000000e+0

**Results**: The learned environment is robust up to the wind speed of 0.4 m/s

In [ ]:
env = gym.make('MultiRobotEnv-v0', render_mode='human')
env.metadata['render_fps'] = 1
obs, info = env.reset()
env.render()
pygame.event.get()

[]

In [23]:
env.close()